# Debiasing
Defining Variables needed for all methods

In [4]:
elangs = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
dlangs = ["mt_MT"]
btypes = ["gender", "race-color", "religion"]

import nltk
nltk.download('punkt_tab')

def get_dataset_path(lang, big_set = False)-> str:
    if lang == "mt_MT":
        return "data/text/mt_MT_100pct.txt"
    elif lang == "es_AR":
        if big_set:
            return "data/text/es_ES_10pct.txt"
        else:
            return "data/text/es_ES_2.5pct.txt"
    elif big_set and lang == "en_US":
        return "data/text/en_US_5pct.txt"
    
    if big_set:
        return f"data/text/{lang}_10pct.txt"
    return f"data/text/{lang}_2.5pct.txt"

def get_bias_attribute_path(lang):
    return f"data/bias_attribute/cleaned/{lang}.json"

def get_crows_path(lang):
    if lang == "es_ES":
        lang = "es_AR"
    return f"data/crows_improved/crows_{lang}.csv"

[nltk_data] Downloading package punkt_tab to /home/jonas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## SentDebias & DensRay

In [ ]:
from experiments.modules.calculate_debias_subspace import SentenceDebiasWrapper, DensrayDebiasWrapper
sent_runner = SentenceDebiasWrapper(save_result=True)
dens_runner = DensrayDebiasWrapper(save_result=True)

for dlang in dlangs:
    print(f"Debiasing with {dlang}")
    sent_runner.setup_data(
        path_to_bias_attributes=get_bias_attribute_path(dlang),
        lang_debias=dlang,
        path_to_dataset=get_dataset_path(dlang),
    )
    sent_runner.compute_gender_subspace()
    sent_runner.compute_racecolor_subspace()
    sent_runner.compute_religion_subspace()
    
    dens_runner.setup_data(
        path_to_bias_attributes=get_bias_attribute_path(dlang),
        lang_debias=dlang,
        path_to_dataset=get_dataset_path(dlang),
    )
    dens_runner.compute_gender_subspace()
    

## INLP

In [ ]:
from experiments.modules.inlp_runner import InlpRunner
inlp_runner = InlpRunner(
    model_class="BertModel",
    model_name_or_path="bert-base-multilingual-uncased",
    save_result=True,
)

for dlang in dlangs:
    print(f"Debiasing with {dlang}")
    for btype in btypes:
        inlp_runner.setup_data(
            path_to_bias_attributes=get_bias_attribute_path(dlang),
            lang_debias=dlang,
            path_to_dataset=get_dataset_path(dlang),
            bias_type=btype,
        )
        inlp_runner.compute_projection_matrix()

## CDA & Dropout

In [ ]:
print("Importing libs...")
from experiments.modules.debias_trainer import CDATrainer, DropoutTrainer
from experiments.modules.crows_runner import CrowSPairsRunnerWrapper
print("Create CrowSRunner...")
crows_runner = CrowSPairsRunnerWrapper(batched=True,)
print("Create DropoutTrainer...")
dropout_trainer = DropoutTrainer(
    model_name_or_path="bert-base-multilingual-uncased",
    fp16=True,
    evaluator_func=lambda model, bias_type, debias_lang: (
        crows_runner.run_debias(
            path_to_crows=get_crows_path(dlang),
            lang_debias=debias_lang,
            lang_eval=debias_lang,
            bias_type=bias_type,
            debias_model=model,  
        )[0]
    )
)
print("Create CDATrainer...")
cda_runner = CDATrainer(
    model_name_or_path="bert-base-multilingual-uncased",
    fp16=True,
    evaluator_func=lambda model, bias_type, debias_lang: (
        crows_runner.run_debias(
            path_to_crows=get_crows_path(dlang),
            lang_debias=debias_lang,
            lang_eval=debias_lang,
            bias_type=bias_type,
            debias_model=model,  
        )[0]
    )
)
print("Staring loop...")
for dlang in dlangs:
    print(f"{dlang}:")
    for btype in btypes:
        cda_runner.train(
            train_file=get_dataset_path(dlang, True),
            bias_attribute_json=get_bias_attribute_path(dlang),
            per_device_train_batch_size=24,
            learning_rate=3e-5,
            warmup_steps=10,
            early_stopping_patience=10,
            debias_lang=dlang,
            bias_type=btype,
            compute_parallel=True,
            save_steps=9000000,
            logging_steps=10,
        )
    dropout_trainer.train(
        train_file=get_dataset_path(dlang, True),
        per_device_train_batch_size=24,
        learning_rate=3e-5,
        warmup_steps=10,
        early_stopping_patience=10,
        debias_lang=dlang,
        compute_parallel=True,
        save_steps=90000000,
    )


Importing libs...
Create CrowSRunner...
Create DropoutTrainer...
Create CDATrainer...
Staring loop...
mt_MT:


/home/jonas/miniconda3/envs/clean312/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 